# 00 — Data Exploration

Load and visualise the Valencia Urban Atlas 2006 data used in the
[IndiFrag tutorial](../reference/IndiFrag_Tutorial_EN.pdf).

**Three input layers:**

| Layer | File | Description |
|-------|------|-------------|
| Objects | `ES003L2_VALENCIA_UA2006_Revised_Clipped_to_Core` | LULC polygons (Urban Atlas 2006) |
| Super-Objects | `VALENCIA_DISTR` | District boundaries (7 districts) |
| Centre Point | `VALENCIA_Centre_Point` | Urban centre reference point |

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path

DATA_DIR = Path("../data/valencia")

## 1. Load the three layers

In [ ]:
objects = gpd.read_file(DATA_DIR / "ES003L2_VALENCIA_UA2006_Revised_Clipped_to_Core.shp")
districts = gpd.read_file(DATA_DIR / "VALENCIA_DISTR.shp")
centre = gpd.read_file(DATA_DIR / "VALENCIA_Centre_Point.shp")

print(f"Objects:   {objects.shape[0]} polygons, CRS = {objects.crs}")
print(f"Districts: {districts.shape[0]} polygons, CRS = {districts.crs}")
print(f"Centre:    {centre.shape[0]} point,    CRS = {centre.crs}")

## 2. Align CRS

Objects are in **EPSG:3035** (ETRS89-LAEA Europe) while districts and centre are
in **EPSG:25830** (ETRS89 / UTM zone 30N).  We reproject everything to UTM 30N
so that area and distance calculations are in metres.

In [ ]:
TARGET_CRS = "EPSG:25830"  # UTM zone 30N — metres

objects = objects.to_crs(TARGET_CRS)
districts = districts.to_crs(TARGET_CRS)
centre = centre.to_crs(TARGET_CRS)

print("All layers now in", TARGET_CRS)
print("Objects bounds (m):", objects.total_bounds.round(0))

## 3. Inspect the object attributes

In [ ]:
objects.drop(columns="geometry").head(10)

## 4. LULC classification

The shapefile already has a `CLASS` field (integers 1-8) mapped from `CODE2006`.
Below is the full lookup, matching the IndiFrag tutorial.

In [ ]:
class_mapping = pd.read_csv(DATA_DIR / "class_mapping.csv")
class_mapping

In [ ]:
# Build a lookup from class_id to class_name
CLASS_NAMES = dict(zip(class_mapping["class_id"], class_mapping["class_name"]))
objects["class_name"] = objects["CLASS"].map(CLASS_NAMES)

print("Objects per class:")
print(objects.groupby(["CLASS", "class_name"]).size().rename("count").reset_index().to_string(index=False))

## 5. Name the districts (super-objects)

The shapefile identifies districts by `CUDIS` code.  The IndiFrag tutorial uses
their local names.

In [ ]:
DISTRICT_NAMES = {
    "4625001": "Ciutat Vella",
    "4625002": "Eixample",
    "4625003": "Extramurs",
    "4625007": "L'Olivereta",
    "4625008": "Patraix",
    "4625009": "Jesús",
    "4625010": "Quatre Carreres",
}
districts["name"] = districts["CUDIS"].map(DISTRICT_NAMES)

districts[["CUDIS", "name", "Shape_area"]].rename(
    columns={"Shape_area": "area_m2"}
)

## 6. Map — LULC objects by class

Reproduces Figure 1 of the IndiFrag tutorial.

In [ ]:
# Colour palette matching the Urban Atlas / IndiFrag tutorial legend
CLASS_COLORS = {
    1: "#ffffbe",  # Agricultural
    2: "#e6a600",  # Barrenland
    3: "#ff8c00",  # Commercial
    4: "#a3d774",  # Green Urban Areas
    5: "#267300",  # Leisure Facilities
    6: "#c8374b",  # Residential
    7: "#cccccc",  # Roads
    8: "#73b2ff",  # Water
}

fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Plot objects coloured by CLASS
objects.plot(
    ax=ax,
    color=objects["CLASS"].map(CLASS_COLORS),
    edgecolor="#888888",
    linewidth=0.15,
)

# Overlay district boundaries
districts.boundary.plot(ax=ax, color="black", linewidth=1.5)

# Centre point
centre.plot(ax=ax, color="red", markersize=80, marker="D", zorder=5)

# Legend
legend_elements = [
    Line2D([0], [0], marker="s", color="w", markerfacecolor=c, markersize=10, label=CLASS_NAMES[k])
    for k, c in CLASS_COLORS.items()
] + [
    Line2D([0], [0], color="black", linewidth=1.5, label="Super-Objects"),
    Line2D([0], [0], marker="D", color="w", markerfacecolor="red", markersize=8, label="Centre Point"),
]
ax.legend(handles=legend_elements, loc="upper right", fontsize=9, framealpha=0.9)

ax.set_title("Valencia — Urban Atlas 2006 (LULC objects by class)", fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.savefig("../outputs/figures/valencia_lulc_overview.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Map — Districts labelled

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

districts.plot(ax=ax, facecolor="#f0f0f0", edgecolor="black", linewidth=1.5)

for _, row in districts.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(
        row["name"],
        xy=(centroid.x, centroid.y),
        ha="center", va="center",
        fontsize=9, fontweight="bold",
    )

centre.plot(ax=ax, color="red", markersize=80, marker="D", zorder=5)

ax.set_title("Valencia — District boundaries (Super-Objects)", fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 8. Basic geometry statistics

Quick sanity check: compute areas in km² and compare with the expected results.

In [ ]:
# Object-level
objects["area_km2"] = objects.geometry.area / 1e6
objects["perim_km"] = objects.geometry.length / 1e3

print("Total study area:  {:.2f} km²".format(objects["area_km2"].sum()))
print("Total objects:     {}".format(len(objects)))
print("Number of classes: {}".format(objects["CLASS"].nunique()))
print()
print("Area per class (km²):")
print(
    objects.groupby("class_name")["area_km2"]
    .agg(["sum", "count", "mean"])
    .round(4)
    .sort_values("sum", ascending=False)
    .rename(columns={"sum": "total_km2", "count": "n_objects", "mean": "mean_km2"})
)

In [ ]:
# District-level areas
districts["area_km2"] = districts.geometry.area / 1e6
districts["perim_km"] = districts.geometry.length / 1e3

districts[["CUDIS", "name", "area_km2", "perim_km"]].sort_values("CUDIS")

## 9. Intersect objects with districts

IndiFrag assigns each object to its super-object. Let's preview this spatial join.
Objects that straddle a district boundary get clipped (overlay/intersection) —
for now we do a simple centroid-based join to count objects per district.

In [ ]:
obj_centroids = objects.copy()
obj_centroids["geometry"] = obj_centroids.geometry.centroid

joined = gpd.sjoin(obj_centroids, districts[["CUDIS", "name", "geometry"]], how="left", predicate="within")

print("Objects per district (centroid join):")
print(joined.groupby("name").size().sort_index().rename("n_objects").to_frame())

## 10. Compare with expected IndiFrag results

Sanity-check: the super-object level results from IndiFrag give `NobSO` (total
objects per district) and `AreaSO` (km²).  Let's see how close we are.

In [ ]:
expected_so = pd.read_csv(
    "../reference/expected_results/Results_FI_SO.txt",
    sep="\t",
    skiprows=4,  # skip header lines
)
expected_so.columns = expected_so.columns.str.strip()
expected_so

In [ ]:
# Compare our centroid-join counts vs IndiFrag's NobSO
# IndiFrag intersects objects with districts (splitting polygons at boundaries),
# so it produces MORE objects per district than our simple centroid assignment.
comparison = (
    joined.groupby("name").size().rename("centroid_join")
    .to_frame()
    .join(
        expected_so.set_index(
            expected_so["CUDIS"].astype(int).astype(str).map(DISTRICT_NAMES)
        )[["NobSO", "AreaSO"]]
    )
)
comparison.index.name = "district"
comparison["NobSO"] = comparison["NobSO"].astype(int)
comparison["diff"] = comparison["NobSO"] - comparison["centroid_join"]
print("Centroid join vs IndiFrag intersection (NobSO):")
print(comparison)
print()
print("The difference is expected: IndiFrag clips objects at district boundaries,")
print("creating extra polygon fragments. Notebook 01 will do a proper intersection.")

---

**Next:** In [01_fragmentation_with_SO.ipynb](./01_fragmentation_with_SO.ipynb) we
implement the full set of fragmentation metrics using open-source Python, following
IndiFrag Tutorial 1.